# My Web Deal Buddy

**What this does:** You describe what you want to buy (and where). **Deals Officer** runs web search and collects prices, merchants, and URLs from snippets. **Deals Manager** turns that into a typed **`DealFinderResult`** (up to 3 ranked deals, plus alternatives, prerequisites, and caveats). **Email Sender** writes a Markdown summary, converts it to HTML, and sends it with **SendGrid** (with a fixed disclaimer). The run is wrapped in **`trace()`** for the OpenAI traces UI.

**You need:** `OPENAI_API_KEY`, `SENDGRID_API_KEY`, and configured **from / to** email in the config cell.

**Learning goals (one per section):**
1. **Agent handoffs** — how one agent passes context to the next
2. **Structured `output_type`** — forcing an agent to return a typed Pydantic model
3. **`function_tool` side effects** — wrapping a real-world action (email) as a tool

**Pipeline:** `Deals Officer` (web search) → `Deals Manager` (structured ranking) → `Email Sender` (email)

## Cell 1 — Imports and config

In [ ]:
!uv pip install markdown

In [ ]:
import os
from typing import Dict

import sendgrid
from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.model_settings import ModelSettings
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv(override=True)

MODEL           = "gpt-4o-mini"

FROM_EMAIL = "noreply@example.com"
TO_EMAIL   = "me@gmail.com"

SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY")
if not SENDGRID_API_KEY:
    raise ValueError("SENDGRID_API_KEY is not set. Add it to .env to send email.")


## Cell 2 — Structured output Schema

In [ ]:
class Deal(BaseModel):
    title:              str = Field(description="Product title grounded in search evidence")
    merchant_or_domain: str = Field(description="Store or domain name if found in evidence, else 'unknown'")
    price_text:         str = Field(description="Price as stated in evidence, else 'unknown'")
    url:                str = Field(description="URL only if present in evidence, else 'unknown'")
    why_ranked:         str = Field(description="One sentence tied to evidence — why this beat the others")


class DealFinderResult(BaseModel):
    ranked_deals:  list[Deal] = Field(description="Up to 3 best options, best first", max_length=3)
    alternatives:  list[str]  = Field(description="Close substitutes or different price tiers")
    prerequisites: list[str]  = Field(description="Accessories, compatibility notes, required subscriptions")
    caveats:       str        = Field(description="Shipping, region limits, price staleness warning")

## Cell 3 — Email tool

In [ ]:
import markdown
from sendgrid.helpers.mail import Mail, Email, To, Content

@function_tool
def send_email(to_email: str, subject: str, body_markdown: str) -> Dict[str, str]:
    """Send deal summary: Markdown body is converted to HTML; disclaimer appended first. Requires SENDGRID_API_KEY."""
    disclaimer = "\n\n---\nPrices and availability are not guaranteed. Verify on the retailer site before purchasing."
    html_body = markdown.markdown(body_markdown + disclaimer)
    
    sg = sendgrid.SendGridAPIClient(api_key=SENDGRID_API_KEY)
    content = Content("text/html", html_body)
    mail = Mail(Email(FROM_EMAIL), To(to_email), subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "sent", "to": to_email}

## Cell 4 — Three agents and the handoff chain

In [ ]:

EMAILER_INSTRUCTIONS = f"""
    You format the deal summary as clear Markdown (headings, bullet lists). 
    Call send_email exactly once with to_email={TO_EMAIL!r}, a short subject line, 
    and body_markdown your summary. The tool converts Markdown to HTML and adds the legal disclaimer.
"""

DEALS_MANAGER_INSTRUCTIONS = """
    "You receive web search notes from the Deals Officer. "
    Rank up to 3 deals using only prices, URLs, and merchants found in those notes. 
    If no deal is provided, compose a response to the user explaining that no deals were found.
    Return a  response that matches the DealFinderResult schema, then hand off to the Email Sender.
"""

DEALS_OFFICER_INSTRUCTIONS = """
    You help shoppers find purchase options. 
    Run 2 to 10 focused web searches (product name + buy + price; add region if given). 
    Summarise what you found — prices, merchants, URLs from snippets only. 
    If you find a good deal, stop searching. When done, hand off to the Deals Manager with your notes.
"""

email_agent = Agent(
    name="Email Sender",
    instructions=EMAILER_INSTRUCTIONS,
    tools=[send_email],
    model=MODEL,
)


deals_manager_agent = Agent(
    name="Deals Manager",
    instructions=DEALS_MANAGER_INSTRUCTIONS,
    output_type=DealFinderResult,
    handoffs=[email_agent],
    model=MODEL,
)


deals_officer_agent = Agent(
    name="Deals Officer",
    instructions=DEALS_OFFICER_INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model_settings=ModelSettings(tool_choice="auto"),
    handoffs=[deals_manager_agent],
    model=MODEL,
)

## Cell 5 — Run

In [ ]:
product_ask = "I need to buy an acoustic guitar in Nigeria, find the best price and merchant"

with trace("deal_finder"):
    result = await Runner.run(
        deals_officer_agent,
        f"Shopping request: {product_ask}\nSend summary to: {TO_EMAIL}",
    )
display(Markdown(str(result.final_output)))